# 01. Training notebook: Weather TFT

Цель ноутбука: обучить модель на train-части и сохранить checkpoint в `checkpoints/`.

Главное правило против утечки данных: этот ноутбук не строит test-frame, не скачивает погоду для test-периода и не использует `guests_count` после `TRAIN_END`.

## 0. Dependencies

ВАЖНО: зависимости для yandex cloud datasphere

In [12]:
%cd project/

[Errno 2] No such file or directory: 'project/'
/home/jupyter/project


In [13]:
!ls -la

total 1349
drwxr-xr-x 1 jupyter jupyter      0 May 10 15:41 .
drwxrwxrwx 1 jupyter jupyter      0 May  9 18:26 ..
drwxr-xr-x 1 jupyter jupyter      0 May  9 09:41 .ipynb_checkpoints
-rw-r--r-- 1 jupyter jupyter   5945 May 10 15:41 Untitled.ipynb
drwxr-xr-x 1 jupyter jupyter      0 May 10 08:00 checkpoints
-rw-r--r-- 1 jupyter jupyter 636288 May 10 07:13 clean_guests.csv
-rw-r--r-- 1 jupyter jupyter 702283 May  9 09:36 guests_orig.csv
-rw-r--r-- 1 jupyter jupyter  36044 May  9 09:12 holiday_weather_compare.ipynb
drwxr-xr-x 1 jupyter jupyter      0 May 10 13:36 lightning_logs
drwxr-xr-x 1 jupyter jupyter      0 May 10 13:01 models
drwxr-xr-x 1 jupyter jupyter      0 May 10 13:01 outputs


In [14]:
!pip install -q \
  torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 \
  --index-url https://download.pytorch.org/whl/cu121

!pip install -q lightning==2.3.3 pytorch-forecasting pandas numpy matplotlib scikit-learn


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


## 1. Configuration

In [15]:
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path
import copy
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import lightning.pytorch as pl

from IPython.display import display
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import QuantileLoss


@dataclass
class TFTPipelineConfig:
    """Main configuration for the weather-aware TFT pipeline."""

    target: str = "guests_count"
    time_idx: str = "time_idx"
    group_ids: list[str] = field(default_factory=lambda: ["restaurant", "segment_id"])

    # 16 business hours per day * 42 days = 672 encoder steps.
    max_encoder_length: int = 16 * 42
    # 16 business hours per day * 7 days = 112 prediction steps.
    max_prediction_length: int = 16 * 7

    batch_size: int = 64
    num_workers: int = 0

    normalizer_group_ids: list[str] = field(default_factory=lambda: ["restaurant"])
    seed: int = 42
    max_epochs: int = 3
    gradient_clip_val: float = 0.1
    accelerator: str = "auto"
    devices: str | int | list[int] = "auto"

    learning_rate: float = 0.03
    hidden_size: int = 32
    attention_head_size: int = 4
    dropout: float = 0.1
    hidden_continuous_size: int = 16
    optimizer: str = "adam"

    checkpoint_dir: str = "checkpoints/present"
    checkpoint_name: str = "tft_weather_model.ckpt"
    metadata_name: str = "tft_weather_model_metadata.json"
    log_dir: str = "lightning_logs"
    output_dir: str = "outputs"
    run_name: str = "tft_weather"

    # Moscow coordinates from the original notebook.
    latitude: float = 55.8018477029042
    longitude: float = 37.53210649745259
    weather_timezone: str = "Europe/Moscow"

    business_start_hour: int = 7
    business_end_hour: int = 22

    static_categoricals: list[str] = field(default_factory=lambda: ["restaurant"])


config = TFTPipelineConfig()


BASELINE_KNOWN_CATEGORICALS = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
]

BASELINE_KNOWN_REALS = [
    "day_of_month",
    "weekofyear",
    "year",
]

HOLIDAY_KNOWN_CATEGORICALS = [
    "is_holiday",
    "is_preholiday",
    "is_shortened_day",
    "is_official_day_off",
    "is_working_weekend",
    "is_transferred_day_off",
    "is_moscow_city_holiday",
    "is_may_holidays_period",
    "production_day_type",
    "holiday_name",
]

HOLIDAY_KNOWN_REALS = [
    "long_weekend_day_number",
    "hour_x_holiday",
    "hour_x_preholiday",
    "hour_x_official_day_off",
    "hour_x_working_weekend",
]

WEATHER_KNOWN_CATEGORICALS = [
    "weather_code",
    "is_day",
    "has_precipitation",
    "has_rain",
    "has_snowfall",
    "is_cloudy",
    "is_windy",
    "precipitation_level",
    "temperature_bin",
]

WEATHER_KNOWN_REALS = [
    "temperature_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "hour_x_precipitation",
    "hour_x_temperature",
]

WEATHER_COLS = [
    "temperature_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "wind_gusts_10m",
    "weather_code",
    "is_day",
]

DEFAULT_WEATHER_VALUES = {
    "temperature_2m": 10.0,
    "apparent_temperature": 10.0,
    "precipitation": 0.0,
    "rain": 0.0,
    "snowfall": 0.0,
    "relative_humidity_2m": 60.0,
    "cloud_cover": 50.0,
    "wind_speed_10m": 3.0,
    "wind_gusts_10m": 6.0,
    "weather_code": 0.0,
    "is_day": 1.0,
}


@dataclass
class TFTFeatureConfig:
    static_categoricals: list[str]
    time_varying_known_categoricals: list[str]
    time_varying_known_reals: list[str]
    time_varying_unknown_categoricals: list[str]
    time_varying_unknown_reals: list[str]


def dedupe(items: list[str]) -> list[str]:
    return list(dict.fromkeys(items))


def build_weather_feature_config(config: TFTPipelineConfig) -> TFTFeatureConfig:
    known_categoricals = (
        BASELINE_KNOWN_CATEGORICALS
        + HOLIDAY_KNOWN_CATEGORICALS
        + WEATHER_KNOWN_CATEGORICALS
    )

    known_reals = (
        BASELINE_KNOWN_REALS
        + HOLIDAY_KNOWN_REALS
        + WEATHER_KNOWN_REALS
    )

    return TFTFeatureConfig(
        static_categoricals=config.static_categoricals,
        time_varying_known_categoricals=dedupe(known_categoricals),
        time_varying_known_reals=dedupe(known_reals),
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=[config.target],
    )


def load_raw_guests(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(path)

    required = {"sale_date", "sale_hour", "guests_count"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise ValueError(f"Missing required input columns: {missing}")

    df = df.copy()
    df["sale_date"] = pd.to_datetime(df["sale_date"]).dt.normalize()
    df["sale_hour"] = pd.to_numeric(df["sale_hour"], errors="raise").astype(int)
    df["guests_count"] = pd.to_numeric(df["guests_count"], errors="coerce")
    df["timestamp"] = df["sale_date"] + pd.to_timedelta(df["sale_hour"], unit="h")

    # Keep one row per business timestamp.
    df = (
        df.groupby("timestamp", as_index=False)["guests_count"]
        .sum()
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    df["sale_date"] = df["timestamp"].dt.normalize()
    df["sale_hour"] = df["timestamp"].dt.hour

    return df


def make_business_hour_grid(
    min_date: pd.Timestamp,
    max_date: pd.Timestamp,
    config: TFTPipelineConfig,
) -> pd.DataFrame:
    dates = pd.date_range(
        pd.Timestamp(min_date).normalize(),
        pd.Timestamp(max_date).normalize(),
        freq="D",
    )

    hours = range(config.business_start_hour, config.business_end_hour + 1)
    rows = [d + pd.Timedelta(hours=h) for d in dates for h in hours]

    grid = pd.DataFrame({"timestamp": rows})
    grid["sale_date"] = grid["timestamp"].dt.normalize()
    grid["sale_hour"] = grid["timestamp"].dt.hour

    return grid


def align_history_to_business_hours(
    raw_df: pd.DataFrame,
    config: TFTPipelineConfig,
    gap_threshold: str = "14D",
) -> pd.DataFrame:
    df = raw_df.copy()
    df = df[
        df["sale_hour"].between(config.business_start_hour, config.business_end_hour)
    ].copy()

    df = df.sort_values("timestamp").reset_index(drop=True)
    gap = df["timestamp"].diff()
    df["segment_id"] = (gap > pd.Timedelta(gap_threshold)).cumsum().astype(int)
    df["segment_id"] = "segment_" + df["segment_id"].astype(str)

    parts = []
    for segment_id, segment in df.groupby("segment_id", sort=True):
        grid = make_business_hour_grid(
            segment["sale_date"].min(),
            segment["sale_date"].max(),
            config,
        )
        grid["segment_id"] = segment_id

        segment_aligned = grid.merge(
            segment[["timestamp", "guests_count"]],
            on="timestamp",
            how="left",
        )
        # Fill missing business hours inside a normal continuous segment.
        segment_aligned["guests_count"] = segment_aligned["guests_count"].fillna(0.0)
        parts.append(segment_aligned)

    if not parts:
        raise ValueError("No rows remain after filtering to business hours.")

    out = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["segment_id", "timestamp"])
        .reset_index(drop=True)
    )

    out["sale_date"] = out["timestamp"].dt.normalize()
    out["sale_hour"] = out["timestamp"].dt.hour

    return out


def make_future_frame(
    start: str | pd.Timestamp,
    periods: int,
    config: TFTPipelineConfig,
) -> pd.DataFrame:
    start_date = pd.Timestamp(start).normalize()
    hours_per_day = config.business_end_hour - config.business_start_hour + 1
    days = int(np.ceil(periods / hours_per_day))

    future = make_business_hour_grid(
        start_date,
        start_date + pd.Timedelta(days=days - 1),
        config,
    )
    future = future.head(periods).copy()
    future["guests_count"] = np.nan

    return future


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["sale_date"] = pd.to_datetime(df["sale_date"]).dt.normalize()

    df["date"] = df["timestamp"].dt.date
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["day_of_month"] = df["timestamp"].dt.day
    df["month"] = df["timestamp"].dt.month
    df["year"] = df["timestamp"].dt.year
    df["weekofyear"] = df["timestamp"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

    return df


FEDERAL_HOLIDAY_NAMES = {
    (1, 1): "new_year",
    (1, 2): "new_year_holidays",
    (1, 3): "new_year_holidays",
    (1, 4): "new_year_holidays",
    (1, 5): "new_year_holidays",
    (1, 6): "new_year_holidays",
    (1, 7): "christmas",
    (1, 8): "new_year_holidays",
    (2, 23): "defender_of_fatherland_day",
    (3, 8): "international_womens_day",
    (5, 1): "spring_and_labor_day",
    (5, 9): "victory_day",
    (6, 12): "russia_day",
    (11, 4): "national_unity_day",
}

MOSCOW_CITY_HOLIDAYS = {
    # Moscow City Day is usually celebrated on the first or second September weekend.
    # It is not an official production-calendar day off, but for restaurant demand it can matter.
    # Add exact dated local events here if you want to model them explicitly.
}

ISDAYOFF_CODES = {
    "0": "working_day",
    "1": "day_off",
    "2": "shortened_day",
    "4": "day_off", # covid legacy
    "8": "holiday",
}

# Offline snapshot of Russian 5-day production calendar overrides.
# Source structure mirrors isdayoff/calendars JSON: dayoff, predayoff, workday, holiday.
# For years not listed here the fallback still uses weekends + fixed federal holidays.
OFFICIAL_RU_CALENDAR_OVERRIDES = {
    2023: {
        "dayoff": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0224", "0308", "0501", "0508", "0509", "0612", "1106"],
        "predayoff": ["0222", "0307", "1103"],
        "workday": [],
        "holiday": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0308", "0501", "0509", "0612", "1104"],
    },
    2024: {
        "dayoff": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0308", "0429", "0430", "0501", "0509", "0510", "0612", "1104", "1230", "1231"],
        "predayoff": ["0222", "0307", "0508", "0611", "1102"],
        "workday": ["0427", "1102", "1228"],
        "holiday": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0308", "0501", "0509", "0612", "1104"],
    },
    2025: {
        "dayoff": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0501", "0502", "0508", "0509", "0612", "0613", "1103", "1104", "1231"],
        "predayoff": ["0307", "0430", "0611", "1101"],
        "workday": ["1101"],
        "holiday": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0308", "0501", "0509", "0612", "1104"],
    },
    2026: {
        "dayoff": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0109", "0110", "0111", "0223", "0309", "0501", "0511", "0612", "1104", "1231"],
        "predayoff": ["0430", "0508", "0611", "1103"],
        "workday": [],
        "holiday": ["0101", "0102", "0103", "0104", "0105", "0106", "0107", "0108", "0223", "0308", "0501", "0509", "0612", "1104"],
    },
}


def _download_isdayoff_period(date_min: pd.Timestamp, date_max: pd.Timestamp) -> pd.DataFrame | None:
    """
    Download official Russian production calendar statuses from isdayoff.ru.

    Status codes from the API:
      0 = working day
      1 = non-working day
      2 = shortened working day
      4 = special non-working day
      8 = public holiday, when holiday=1 is requested

    The calendar is a known exogenous feature: it is safe to use for future dates,
    unlike actual weather or target values.
    """
    date_min = pd.Timestamp(date_min).normalize()
    date_max = pd.Timestamp(date_max).normalize()
    if pd.isna(date_min) or pd.isna(date_max) or date_min > date_max:
        return None

    cache_dir = Path("cache") / "production_calendar"
    cache_dir.mkdir(parents=True, exist_ok=True)

    frames = []
    for year in range(date_min.year, date_max.year + 1):
        cache_path = cache_dir / f"isdayoff_ru_{year}.csv"
        if cache_path.exists():
            frames.append(pd.read_csv(cache_path, parse_dates=["sale_date"]))
            continue

        try:
            response = requests.get(
                "https://isdayoff.ru/api/getdata",
                params={
                    "year": year,
                    "cc": "ru",
                    "pre": 1,
                    "covid": 1,
                    "delimeter": "%0A",
                },
                timeout=30,
            )
            response.raise_for_status()
            codes = [x.strip() for x in response.text.splitlines() if x.strip()]
            dates = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
            if len(codes) != len(dates):
                raise ValueError(f"Unexpected isdayoff length for {year}: {len(codes)} vs {len(dates)}")

            year_frame = pd.DataFrame({"sale_date": dates, "isdayoff_code": codes})
            year_frame.to_csv(cache_path, index=False)
            frames.append(year_frame)
        except Exception as exc:
            warnings.warn(
                f"Could not download official production calendar for {year}: {exc}. "
                "Using deterministic fallback based on weekends and fixed federal holidays."
            )
            return None

    calendar = pd.concat(frames, ignore_index=True)
    mask = calendar["sale_date"].between(date_min, date_max)
    return calendar.loc[mask].copy()


def _fallback_production_calendar(date_min: pd.Timestamp, date_max: pd.Timestamp) -> pd.DataFrame:
    """
    Offline fallback.

    For 2023-2026 it uses an embedded official-production-calendar snapshot.
    For other years it falls back to weekends + fixed federal holidays.
    """
    dates = pd.date_range(pd.Timestamp(date_min).normalize(), pd.Timestamp(date_max).normalize(), freq="D")
    frame = pd.DataFrame({"sale_date": dates})
    frame["isdayoff_code"] = "0"

    for idx, row in frame.iterrows():
        current = row["sale_date"]
        mmdd = current.strftime("%m%d")
        year_calendar = OFFICIAL_RU_CALENDAR_OVERRIDES.get(current.year)

        if year_calendar is not None:
            if mmdd in year_calendar.get("workday", []):
                code = "0"
            elif mmdd in year_calendar.get("predayoff", []):
                code = "2"
            elif mmdd in year_calendar.get("dayoff", []):
                code = "1"
            elif current.dayofweek >= 5:
                code = "1"
            else:
                code = "0"
        else:
            fixed_holiday = (current.month, current.day) in FEDERAL_HOLIDAY_NAMES
            next_day = current + pd.Timedelta(days=1)
            preholiday = (next_day.month, next_day.day) in FEDERAL_HOLIDAY_NAMES
            if fixed_holiday or current.dayofweek >= 5:
                code = "1"
            elif preholiday:
                code = "2"
            else:
                code = "0"

        frame.at[idx, "isdayoff_code"] = code

    return frame


def build_production_calendar(date_min: pd.Timestamp, date_max: pd.Timestamp) -> pd.DataFrame:
    calendar = _download_isdayoff_period(date_min, date_max)
    if calendar is None:
        calendar = _fallback_production_calendar(date_min, date_max)

    calendar = calendar.copy()
    calendar["sale_date"] = pd.to_datetime(calendar["sale_date"]).dt.normalize()
    calendar["isdayoff_code"] = calendar["isdayoff_code"].astype(str)
    calendar["production_day_type"] = calendar["isdayoff_code"].map(ISDAYOFF_CODES).fillna("unknown")
    return calendar[["sale_date", "isdayoff_code", "production_day_type"]]


def add_holiday_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["sale_date"] = pd.to_datetime(df["sale_date"]).dt.normalize()
    df["month"] = df["sale_date"].dt.month
    df["day_of_month"] = df["sale_date"].dt.day
    df["day_of_week"] = df["sale_date"].dt.dayofweek

    calendar = build_production_calendar(df["sale_date"].min(), df["sale_date"].max())
    df = df.merge(calendar, on="sale_date", how="left")
    df["isdayoff_code"] = df["isdayoff_code"].fillna("0").astype(str)
    df["production_day_type"] = df["production_day_type"].fillna("working_day")

    m = df["month"]
    d = df["day_of_month"]
    weekday = df["day_of_week"]

    fixed_holiday_name = df["sale_date"].map(
        lambda x: FEDERAL_HOLIDAY_NAMES.get((x.month, x.day), "none")
    )
    is_fixed_federal_holiday = fixed_holiday_name.ne("none")
    is_weekend = weekday >= 5

    df["is_official_day_off"] = df["isdayoff_code"].isin(["1", "4", "8"]).astype(int)
    df["is_holiday"] = (df["is_official_day_off"].eq(1) | is_fixed_federal_holiday).astype(int)
    df["is_shortened_day"] = df["isdayoff_code"].eq("2").astype(int)
    df["is_preholiday"] = df["is_shortened_day"].copy()
    df["is_working_weekend"] = (is_weekend & df["isdayoff_code"].isin(["0", "2"])).astype(int)
    df["is_transferred_day_off"] = (
        df["is_official_day_off"].eq(1) & (~is_weekend) & (~is_fixed_federal_holiday)
    ).astype(int)

    df["is_new_year_holidays"] = ((m == 1) & d.between(1, 8)).astype(int)
    df["is_dec_31"] = ((m == 12) & (d == 31)).astype(int)
    df["is_jan_1"] = ((m == 1) & (d == 1)).astype(int)
    df["is_jan_2_to_8"] = ((m == 1) & d.between(2, 8)).astype(int)
    df["is_feb_23"] = ((m == 2) & (d == 23)).astype(int)
    df["is_mar_8"] = ((m == 3) & (d == 8)).astype(int)
    df["is_may_1"] = ((m == 5) & (d == 1)).astype(int)
    df["is_may_9"] = ((m == 5) & (d == 9)).astype(int)
    df["is_jun_12"] = ((m == 6) & (d == 12)).astype(int)
    df["is_nov_4"] = ((m == 11) & (d == 4)).astype(int)

    df["holiday_name"] = fixed_holiday_name
    df.loc[df["is_transferred_day_off"].eq(1) & df["holiday_name"].eq("none"), "holiday_name"] = "transferred_day_off"
    df.loc[df["is_working_weekend"].eq(1), "holiday_name"] = "working_weekend"
    df.loc[df["is_shortened_day"].eq(1) & df["holiday_name"].eq("none"), "holiday_name"] = "shortened_day"

    # City-level Moscow events are not official days off, but can be added as demand features.
    df["is_moscow_city_holiday"] = df["sale_date"].map(lambda x: x.strftime("%Y-%m-%d") in MOSCOW_CITY_HOLIDAYS).astype(int)

    may_holidays_period = ((m == 4) & (d >= 29)) | ((m == 5) & (d <= 10))
    df["is_may_holidays_period"] = may_holidays_period.astype(int)

    may_period_dates = (
        df.loc[may_holidays_period, "sale_date"]
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )
    may_day_map = {date: i + 1 for i, date in enumerate(may_period_dates)}
    df["long_weekend_day_number"] = 0
    if len(may_day_map):
        df.loc[may_holidays_period, "long_weekend_day_number"] = (
            df.loc[may_holidays_period, "sale_date"].map(may_day_map).fillna(0).astype(int)
        )

    df["hour_x_holiday"] = df["hour"] * df["is_holiday"]
    df["hour_x_preholiday"] = df["hour"] * df["is_preholiday"]
    df["hour_x_new_year"] = df["hour"] * df["is_new_year_holidays"]
    df["hour_x_may_holidays"] = df["hour"] * df["is_may_holidays_period"]
    df["hour_x_official_day_off"] = df["hour"] * df["is_official_day_off"]
    df["hour_x_working_weekend"] = df["hour"] * df["is_working_weekend"]

    return df

def download_weather(
    min_date: pd.Timestamp,
    max_date: pd.Timestamp,
    config: TFTPipelineConfig,
) -> pd.DataFrame:
    """Download actual historical weather for an already observed date range."""

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": config.latitude,
        "longitude": config.longitude,
        "start_date": pd.Timestamp(min_date).strftime("%Y-%m-%d"),
        "end_date": pd.Timestamp(max_date).strftime("%Y-%m-%d"),
        "hourly": WEATHER_COLS,
        "timezone": config.weather_timezone,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    weather_json = response.json()
    weather = pd.DataFrame(weather_json["hourly"])
    weather["timestamp"] = pd.to_datetime(weather["time"])
    weather = weather.drop(columns=["time"])

    return weather


def add_weather_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for col, default_value in DEFAULT_WEATHER_VALUES.items():
        if col not in df.columns:
            df[col] = default_value
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default_value)

    df["has_precipitation"] = (df["precipitation"] > 0).astype(int)
    df["has_rain"] = (df["rain"] > 0).astype(int)
    df["has_snowfall"] = (df["snowfall"] > 0).astype(int)
    df["is_cloudy"] = (df["cloud_cover"] >= 70).astype(int)
    df["is_windy"] = (df["wind_speed_10m"] >= 8).astype(int)

    df["precipitation_level"] = pd.cut(
        df["precipitation"],
        bins=[-0.01, 0, 1, 3, np.inf],
        labels=["none", "light", "medium", "heavy"],
    ).astype(str)

    df["temperature_bin"] = pd.cut(
        df["temperature_2m"],
        bins=[-50, 0, 10, 18, 25, 50],
        labels=["freezing", "cold", "cool", "warm", "hot"],
    ).astype(str)

    df["hour_x_precipitation"] = df["hour"] * df["has_precipitation"]
    df["hour_x_temperature"] = df["hour"] * df["temperature_2m"]

    return df


def add_actual_weather_to_history(
    history_df: pd.DataFrame,
    config: TFTPipelineConfig,
) -> pd.DataFrame:
    """Attach actual weather only for rows that are already in the past."""

    history = history_df.copy()
    if history.empty:
        raise ValueError("Cannot add weather to an empty history frame.")

    weather = download_weather(
        history["sale_date"].min(),
        history["sale_date"].max(),
        config,
    )

    history = history.merge(weather, on="timestamp", how="left")
    history = add_weather_derived_features(history)

    return history


def fill_future_weather_from_history(
    history_df: pd.DataFrame,
    future_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build weather features for a predicted horizon without downloading that horizon's weather.

    Priority:
      1. median actual weather for the same month-day and hour from past history;
      2. median actual weather for the same month and hour from past history;
      3. conservative defaults.
    """

    history = history_df.copy()
    future = future_df.copy()

    for col, default_value in DEFAULT_WEATHER_VALUES.items():
        if col not in history.columns:
            history[col] = default_value
        history[col] = pd.to_numeric(history[col], errors="coerce").fillna(default_value)

    history["month_day"] = pd.to_datetime(history["sale_date"]).dt.strftime("%m-%d")
    future["month_day"] = pd.to_datetime(future["sale_date"]).dt.strftime("%m-%d")

    by_day_hour = (
        history.groupby(["month_day", "hour"], as_index=False)[WEATHER_COLS]
        .median(numeric_only=True)
    )
    future = future.merge(by_day_hour, on=["month_day", "hour"], how="left")

    by_month_hour = (
        history.groupby(["month", "hour"], as_index=False)[WEATHER_COLS]
        .median(numeric_only=True)
    )
    future = future.merge(
        by_month_hour,
        on=["month", "hour"],
        how="left",
        suffixes=("", "_month"),
    )

    for col, default_value in DEFAULT_WEATHER_VALUES.items():
        month_col = f"{col}_month"
        if month_col in future.columns:
            future[col] = future[col].fillna(future[month_col])
        future[col] = future[col].fillna(default_value)

    drop_cols = ["month_day"] + [
        f"{col}_month" for col in WEATHER_COLS if f"{col}_month" in future.columns
    ]
    future = future.drop(columns=drop_cols)

    return future


def add_base_series_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["restaurant"] = "single_restaurant"
    df["data_missing"] = 0
    df["guests_count_original"] = df["guests_count"]
    return df


def build_observed_calendar_frame(
    raw_df: pd.DataFrame,
    config: TFTPipelineConfig,
) -> pd.DataFrame:
    history = align_history_to_business_hours(raw_df, config)
    history = add_calendar_features(history)
    history = add_holiday_features(history)
    history = add_base_series_columns(history)
    return history


def enforce_tft_dtypes(
    df: pd.DataFrame,
    config: TFTPipelineConfig,
    feature_config: TFTFeatureConfig,
) -> pd.DataFrame:
    df = df.copy()

    categorical_cols = dedupe(
        config.group_ids
        + feature_config.static_categoricals
        + feature_config.time_varying_known_categoricals
        + feature_config.time_varying_unknown_categoricals
    )

    real_cols = dedupe(
        feature_config.time_varying_known_reals
        + feature_config.time_varying_unknown_reals
        + [config.target, config.time_idx]
    )

    for col in categorical_cols:
        if col not in df.columns:
            raise ValueError(f"Categorical column is missing: {col}")
        df[col] = df[col].fillna("missing").astype(str)

    for col in real_cols:
        if col not in df.columns:
            raise ValueError(f"Real column is missing: {col}")
        df[col] = pd.to_numeric(df[col], errors="coerce")

    for col in real_cols:
        if col != config.target:
            df[col] = df[col].fillna(0.0)

    return df


def check_tft_columns(
    df: pd.DataFrame,
    config: TFTPipelineConfig,
    feature_config: TFTFeatureConfig,
) -> None:
    required_cols = dedupe(
        [config.target, config.time_idx]
        + config.group_ids
        + feature_config.static_categoricals
        + feature_config.time_varying_known_categoricals
        + feature_config.time_varying_known_reals
        + feature_config.time_varying_unknown_categoricals
        + feature_config.time_varying_unknown_reals
    )

    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing columns for TFT: {missing}")

    na_share = df[required_cols].isna().mean().sort_values(ascending=False)
    na_share = na_share[na_share > 0]
    if len(na_share):
        raise ValueError(f"NaN found in TFT columns:\n{na_share}")


def build_training_frame(
    raw_df: pd.DataFrame,
    config: TFTPipelineConfig,
    train_end: str,
) -> tuple[pd.DataFrame, TFTFeatureConfig]:
    """
    Build the training frame.

    Leakage contract:
      - target rows are strictly earlier than train_end;
      - actual weather is downloaded only for these training rows;
      - no test-period rows or test-period weather are present in this frame.
    """

    train_end_ts = pd.Timestamp(train_end).normalize()

    observed = build_observed_calendar_frame(raw_df, config)
    train = observed[observed["timestamp"] < train_end_ts].copy()

    if train.empty:
        raise ValueError(f"No training rows before train_end={train_end}.")

    train = add_actual_weather_to_history(train, config)
    train = train.sort_values(["segment_id", "timestamp"]).reset_index(drop=True)
    train[config.time_idx] = np.arange(len(train), dtype=int)

    train[config.target] = pd.to_numeric(train[config.target], errors="coerce").fillna(0.0)

    feature_config = build_weather_feature_config(config)
    train = enforce_tft_dtypes(train, config, feature_config)
    check_tft_columns(train, config, feature_config)

    print("Training leakage audit")
    print("  train_end:", train_end_ts)
    print("  max train timestamp:", train["timestamp"].max())
    print("  weather API range:", train["sale_date"].min(), "..", train["sale_date"].max())
    print("  rows:", len(train))

    return train, feature_config


def make_training_dataset(
    train_df: pd.DataFrame,
    config: TFTPipelineConfig,
    feature_config: TFTFeatureConfig,
) -> tuple[TimeSeriesDataSet, torch.utils.data.DataLoader]:
    all_categoricals = dedupe(
        config.group_ids
        + feature_config.static_categoricals
        + feature_config.time_varying_known_categoricals
        + feature_config.time_varying_unknown_categoricals
    )
    categorical_encoders = {
        col: NaNLabelEncoder(add_nan=True)
        for col in all_categoricals
    }

    training = TimeSeriesDataSet(
        train_df,
        time_idx=config.time_idx,
        target=config.target,
        group_ids=config.group_ids,
        min_encoder_length=config.max_encoder_length // 2,
        max_encoder_length=config.max_encoder_length,
        min_prediction_length=config.max_prediction_length,
        max_prediction_length=config.max_prediction_length,
        static_categoricals=feature_config.static_categoricals,
        time_varying_known_categoricals=feature_config.time_varying_known_categoricals,
        time_varying_known_reals=feature_config.time_varying_known_reals,
        time_varying_unknown_categoricals=feature_config.time_varying_unknown_categoricals,
        time_varying_unknown_reals=feature_config.time_varying_unknown_reals,
        target_normalizer=GroupNormalizer(
            groups=config.normalizer_group_ids,
            transformation="softplus",
        ),
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        allow_missing_timesteps=True,
        categorical_encoders=categorical_encoders,
    )

    train_loader = training.to_dataloader(
        train=True,
        batch_size=config.batch_size,
        num_workers=config.num_workers,
    )

    return training, train_loader

## 2. Train-only parameters

`TRAIN_END` — первая дата test-периода. Все строки с `timestamp >= TRAIN_END` исключаются из обучения и из погодного API-запроса.

In [16]:
CSV_PATH = Path("clean_guests.csv")
TEST_WEEKS = 8
# None = automatically hold out the last 8 complete 7-day windows from clean_guests.csv.
# You can still set a fixed date, e.g. TRAIN_END = "2026-04-20".
TRAIN_END = None

# Optional overrides for a longer real run:
# config.max_epochs = 20
# config.batch_size = 128
# config.learning_rate = 0.01

## 3. Train and save checkpoint

In [17]:
def infer_train_end_for_last_full_test_weeks(
    csv_path: str | Path,
    config: TFTPipelineConfig,
    test_weeks: int = 8,
) -> tuple[str, list[str]]:
    """
    Infer a leakage-safe split: train ends at the first timestamp of the last
    `test_weeks` complete 7-day prediction windows available in the CSV.

    Example: if the last complete business day is Sunday, the first test window
    usually starts on Monday three weeks earlier.
    """
    if test_weeks < 1:
        raise ValueError("test_weeks must be >= 1")

    raw_df = load_raw_guests(csv_path)
    observed = build_observed_calendar_frame(raw_df, config)
    last_timestamp = pd.to_datetime(observed["timestamp"]).max()

    starts: list[pd.Timestamp] = []
    current_start = last_timestamp.normalize() - pd.Timedelta(days=6)

    while len(starts) < test_weeks:
        future = make_future_frame(current_start, config.max_prediction_length, config)
        if future["timestamp"].max() <= last_timestamp:
            starts.append(current_start)
        current_start = current_start - pd.Timedelta(days=7)

        if current_start < observed["timestamp"].min().normalize():
            raise ValueError(
                f"Could not find {test_weeks} complete 7-day test windows in the CSV."
            )

    starts = sorted(starts)
    return str(starts[0].date()), [str(s.date()) for s in starts]


def train_weather_model(
    csv_path: str | Path,
    train_end: str | None,
    config: TFTPipelineConfig = config,
    test_weeks: int = 8,
) -> tuple[pd.DataFrame, str, str]:
    """
    Train TFT only on rows before train_end and save the checkpoint to checkpoints/.
    If train_end is None, the last `test_weeks` complete 7-day windows are held out.
    """

    pl.seed_everything(config.seed, workers=True)

    raw_df = load_raw_guests(csv_path)

    if train_end is None:
        split_strategy = "last_complete_rolling_weeks"
        train_end, planned_test_week_starts = infer_train_end_for_last_full_test_weeks(
            csv_path=csv_path,
            config=config,
            test_weeks=test_weeks,
        )
    else:
        split_strategy = "fixed_train_end"
        planned_test_week_starts = [str(pd.Timestamp(train_end).date())]

    print(f"Train/test split: train < {train_end}; test windows = {planned_test_week_starts}")

    train_df, feature_config = build_training_frame(
        raw_df=raw_df,
        config=config,
        train_end=train_end,
    )

    training, train_loader = make_training_dataset(train_df, config, feature_config)

    tft = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=config.learning_rate,
        hidden_size=config.hidden_size,
        attention_head_size=config.attention_head_size,
        dropout=config.dropout,
        hidden_continuous_size=config.hidden_continuous_size,
        loss=QuantileLoss(),
        optimizer=config.optimizer,
    )

    print(f"Number of parameters: {tft.size() / 1e3:.1f}k")

    trainer = pl.Trainer(
        max_epochs=config.max_epochs,
        accelerator=config.accelerator,
        devices=config.devices,
        gradient_clip_val=config.gradient_clip_val,
        callbacks=[LearningRateMonitor(logging_interval="epoch")],
        logger=TensorBoardLogger(config.log_dir, name=config.run_name),
        enable_checkpointing=False,
        log_every_n_steps=50,
    )

    trainer.fit(tft, train_dataloaders=train_loader)

    checkpoint_dir = Path(config.checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = checkpoint_dir / config.checkpoint_name
    metadata_path = checkpoint_dir / config.metadata_name

    trainer.save_checkpoint(str(checkpoint_path))

    metadata = {
        "checkpoint_path": str(checkpoint_path),
        "train_end": str(pd.Timestamp(train_end).date()),
        "split_strategy": split_strategy,
        "test_weeks": int(test_weeks),
        "planned_test_week_starts": planned_test_week_starts,
        "csv_path": str(csv_path),
        "train_rows": int(len(train_df)),
        "train_min_timestamp": str(train_df["timestamp"].min()),
        "train_max_timestamp": str(train_df["timestamp"].max()),
        "weather_api_max_date": str(pd.to_datetime(train_df["sale_date"]).max().date()),
        "config": asdict(config),
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    print(f"Saved checkpoint: {checkpoint_path}")
    print(f"Saved metadata: {metadata_path}")

    return train_df, str(checkpoint_path), str(metadata_path)

In [18]:
train_df, checkpoint_path, metadata_path = train_weather_model(
    csv_path=CSV_PATH,
    train_end=TRAIN_END,
    config=config,
    test_weeks=TEST_WEEKS,
)

checkpoint_path, metadata_path

Seed set to 42
/tmp/ipykernel_4423/610894814.py:448: UserWarning: Could not download official production calendar for 2019: Unexpected isdayoff length for 2019: 1 vs 365. Using deterministic fallback based on weekends and fixed federal holidays.
  warnings.warn(


Train/test split: train < 2026-03-02; test windows = ['2026-03-02', '2026-03-09', '2026-03-16', '2026-03-23', '2026-03-30', '2026-04-06', '2026-04-13', '2026-04-20']


/tmp/ipykernel_4423/610894814.py:448: UserWarning: Could not download official production calendar for 2019: Unexpected isdayoff length for 2019: 1 vs 365. Using deterministic fallback based on weekends and fixed federal holidays.
  warnings.warn(


Training leakage audit
  train_end: 2026-03-02 00:00:00
  max train timestamp: 2026-03-01 22:00:00
  weather API range: 2019-01-02 00:00:00 .. 2026-03-01 00:00:00
  rows: 35968


/home/jupyter/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/jupyter/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
/home/jupyter/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/jupyter/.local/lib/python3.10/site-packages/lightning/pytorch/utilities/parsing.py:208: Attribute 'logging_metrics' is an instance of `nn.Module` and is already save

Number of parameters: 175.1k


/home/jupyter/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
2026-05-10 18:15:42.559886: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-10 18:15:43.863974: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-10 18:15:47.491228: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name          

Epoch 2:  58%|█████▊    | 323/555 [10:00<07:11,  0.54it/s, v_num=1, train_loss_step=3.930, train_loss_epoch=4.300]

: 

: 

## 4. Quick train-frame sanity checks

In [19]:
print("Train min timestamp:", train_df["timestamp"].min())
print("Train max timestamp:", train_df["timestamp"].max())
print("Rows:", len(train_df))

display(train_df[["timestamp", "guests_count", "temperature_2m", "precipitation", "segment_id"]].tail(20))

Train min timestamp: 2019-01-02 07:00:00
Train max timestamp: 2026-03-01 22:00:00
Rows: 35968


,timestamp,guests_count,temperature_2m,precipitation,segment_id
35948,2026-02-28 19:00:00,136.0,1.6,0.0,segment_3
35949,2026-02-28 20:00:00,129.0,1.7,0.0,segment_3
35950,2026-02-28 21:00:00,102.0,1.8,0.0,segment_3
35951,2026-02-28 22:00:00,75.0,1.7,0.0,segment_3
35952,2026-03-01 07:00:00,13.0,1.1,0.0,segment_3
35953,2026-03-01 08:00:00,34.0,1.0,0.0,segment_3
35954,2026-03-01 09:00:00,44.0,1.3,0.0,segment_3
35955,2026-03-01 10:00:00,47.0,1.4,0.0,segment_3
35956,2026-03-01 11:00:00,60.0,1.6,0.0,segment_3
35957,2026-03-01 12:00:00,77.0,2.0,0.0,segment_3
